# Phase 3: tactical bi-encoder (retrieval of similar shots)




## 0. Setup  [every session, seconds]

In [1]:
import json, random, re, collections
from pathlib import Path
import numpy as np, pandas as pd, torch, torch.nn.functional as F
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device, torch.cuda.get_device_name() if device == "cuda" else "")

DATA, MODELS, RES = Path("../data/processed"), Path("../models"), Path("../results/phase3")
MODELS.mkdir(exist_ok=True); RES.mkdir(parents=True, exist_ok=True)

ATTRS = ["team", "body", "side", "area", "outcome"]    # the 5 attributes a query can ask about
corpus = [json.loads(l) for l in open(DATA / "text_corpus.jsonl", encoding="utf-8")]   # one row per shot, with full doc_attrs
pairs  = [json.loads(l) for l in open(DATA / "text_pairs.jsonl",  encoding="utf-8")]   # 3 template queries per shot
print(len(corpus), "docs |", len(pairs), "pairs |",
      {s: sum(c["split"] == s for c in corpus) for s in ("train", "val", "test")})

c:\Users\avish\OneDrive\Desktop\SEM2\prog_ai2\project\football\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


cuda NVIDIA GeForce RTX 4060 Laptop GPU
10233 docs | 30699 pairs | {'train': 7290, 'val': 1463, 'test': 1480}


## 1. Ground truth

A document is **relevant** to a query if it satisfies *all* the attributes the query asks for (`doc_attrs` holds each document's full attributes). Retrieval is evaluated within one split's corpus (about 1,460 documents). Median query has about 46 relevant documents; 70% have 10 or more, so a plain "any hit in the top k" saturates and we also report nP@10.

In [2]:
def rel_matrix(docs, attr_list):
    return np.array([[all(d["doc_attrs"][k] == v for k, v in a.items()) for d in docs] for a in attr_list])

def build_rel(split):
    docs = [c for c in corpus if c["split"] == split]
    qs   = [p for p in pairs  if p["split"] == split]
    return docs, qs, rel_matrix(docs, [q["attrs"] for q in qs])     # rel: (n_queries, n_docs) bool

val_docs, val_qs, val_rel = build_rel("val")
print(val_rel.shape, "relevant docs per query: min", val_rel.sum(1).min(), "median", int(np.median(val_rel.sum(1))), "mean", val_rel.sum(1).mean().round(1))

(4389, 1463) relevant docs per query: min 1 median 46 mean 103.2


## 2. Metrics

- **Recall@k (R@k):** a relevant document is in the top k. **MRR:** mean of 1/rank of the first relevant document.
- **P@10:** fraction of the top 10 that are relevant. Its ceiling is below 1 when a query has fewer than 10 relevant documents, so **nP@10** divides by `min(10, n_relevant)` and is exactly 1.0 for a perfect ranking.
- Reported for all queries, and split by queries with at most 10 or more than 10 relevant documents.

In [3]:
def evaluate_retrieval(scores, rel, ks=(1, 5, 10)):
    order  = np.argsort(-scores, axis=1)                 # doc indices, best first
    ranked = np.take_along_axis(rel, order, axis=1)      # relevance flags in ranked order
    first  = ranked.argmax(1)                            # column of the first relevant doc
    n_rel  = rel.sum(1)

    def metrics(m):                                      # m = boolean mask over queries
        r = ranked[m]
        out = {f"R@{k}": r[:, :k].any(1).mean() for k in ks}
        out["MRR"]   = (1.0 / (first[m] + 1)).mean()
        out["P@10"]  = r[:, :10].mean()
        out["nP@10"] = (r[:, :10].sum(1) / np.minimum(10, n_rel[m])).mean()
        return out

    return pd.DataFrame({"all": metrics(np.ones(len(rel), bool)),
                         "<=10 rel": metrics(n_rel <= 10),
                         ">10 rel": metrics(n_rel > 10)}).T.round(3)

def per_query_np(scores, rel):
    ranked = np.take_along_axis(rel, np.argsort(-scores, axis=1), axis=1)
    return ranked[:, :10].sum(1) / np.minimum(10, rel.sum(1))

# sanity checks: the oracle must give 1.0 on everything except plain P@10; random is the floor
print(evaluate_retrieval(val_rel.astype(float), val_rel))
print(evaluate_retrieval(np.random.rand(*val_rel.shape), val_rel))

          R@1  R@5  R@10  MRR   P@10  nP@10
all       1.0  1.0   1.0  1.0  0.831    1.0
<=10 rel  1.0  1.0   1.0  1.0  0.466    1.0
>10 rel   1.0  1.0   1.0  1.0  1.000    1.0
            R@1    R@5   R@10    MRR   P@10  nP@10
all       0.073  0.246  0.362  0.163  0.068  0.070
<=10 rel  0.001  0.018  0.035  0.018  0.004  0.008
>10 rel   0.106  0.351  0.514  0.231  0.098  0.098


## 3. Baselines before fine-tuning

In [4]:
# BM25: word matching, no learning. Scores are cached because computing them takes about a minute.
def tok(text): return re.findall(r"\w+", text.lower())

def bm25_scores(docs, qs):
    bm25 = BM25Okapi([tok(d["document"]) for d in docs])
    return np.array([bm25.get_scores(tok(q["query"])) for q in qs])

bm25_path = RES / "bm25_val.npy"
if bm25_path.exists():
    S_bm25 = np.load(bm25_path)
else:
    S_bm25 = bm25_scores(val_docs, val_qs); np.save(bm25_path, S_bm25)
bm25_val = evaluate_retrieval(S_bm25, val_rel)
print(bm25_val)

# where does BM25 struggle? nP@10 for queries that do / do not mention each attribute
npq = per_query_np(S_bm25, val_rel)
for attr in ATTRS:
    has = np.array([attr in q["attrs"] for q in val_qs])
    print(f"{attr:8s} with: {npq[has].mean():.3f}   without: {npq[~has].mean():.3f}")

            R@1    R@5   R@10    MRR   P@10  nP@10
all       0.728  0.953  0.984  0.820  0.639  0.785
<=10 rel  0.609  0.920  0.977  0.735  0.351  0.812
>10 rel   0.783  0.968  0.987  0.859  0.773  0.773
team     with: 0.788   without: 0.782
body     with: 0.755   without: 0.816
side     with: 0.751   without: 0.820
area     with: 0.783   without: 0.788
outcome  with: 0.792   without: 0.779


BM25 is strong on template text (R@10 0.98) because queries and documents share vocabulary. Its weak spot is queries mentioning `body` or `side`: "left" and "right" appear in both ("left foot", "left side") and a bag of words cannot tell them apart.

In [5]:
# Pretrained MiniLM, no fine-tuning: the real "before" for the report.
pre = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device)
pre.max_seq_length = 128
D = pre.encode([d["document"] for d in val_docs], batch_size=128, normalize_embeddings=True, convert_to_tensor=True)
Q = pre.encode([q["query"] for q in val_qs],      batch_size=128, normalize_embeddings=True, convert_to_tensor=True)
pre_val = evaluate_retrieval((Q @ D.T).cpu().numpy(), val_rel)
print(pre_val)

lens = [len(pre.tokenizer(d["document"])["input_ids"]) for d in val_docs]       # does anything get truncated?
print("token length percentiles 50/90/99/100:", np.percentile(lens, [50, 90, 99, 100]), "| limit", pre.max_seq_length)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4370.53it/s]


            R@1    R@5   R@10    MRR   P@10  nP@10
all       0.333  0.605  0.725  0.462  0.281  0.382
<=10 rel  0.336  0.727  0.857  0.506  0.222  0.542
>10 rel   0.331  0.549  0.663  0.441  0.308  0.308
token length percentiles 50/90/99/100: [ 63.  80.  96. 113.] | limit 128


Pretrained MiniLM is far worse than BM25 here (nP@10 0.38 vs 0.79): it knows English but not which details of a shot description matter. The longest document is 113 tokens, so `max_seq_length = 128` truncates nothing.

## 4. Evaluation sets



In [6]:
PARA = [  # (query, attrs)
 ("headers from the left wing the keeper saved",              {"body":"head","side":"left","outcome":"was saved"}),
 ("long-range efforts that went wide",                        {"area":"outside the box","outcome":"missed the target"}),
 ("goals scored inside the penalty area",                     {"area":"inside the box","outcome":"ended in a goal"}),
 ("right-footed strikes from the right that a defender blocked", {"body":"right foot","side":"right","outcome":"was blocked"}),
 ("shots the keeper stopped from the middle of the pitch area",  {"side":"central","outcome":"was saved"}),
 ("left-footed goals",                                        {"body":"left foot","outcome":"ended in a goal"}),
 ("England Women's attempts blocked inside the area",         {"team":"England Women's","area":"inside the box","outcome":"was blocked"}),
 ("Qatar efforts from distance",                              {"team":"Qatar","area":"outside the box"}),
 ("Denmark headers that found the net",                       {"team":"Denmark","body":"head","outcome":"ended in a goal"}),
 ("Spain Women's shots off the right flank that missed",      {"team":"Spain Women's","side":"right","outcome":"missed the target"}),
 ("saved attempts from outside the penalty area on the left", {"area":"outside the box","side":"left","outcome":"was saved"}),
 ("headed chances inside the box",                            {"body":"head","area":"inside the box"}),
 ("right foot shots from the centre that were stopped by the goalkeeper", {"body":"right foot","side":"central","outcome":"was saved"}),
 ("balls hit over the bar from range with the left foot",     {"body":"left foot","area":"outside the box","outcome":"missed the target"}),
 ("central headers that ended up in the net",                 {"body":"head","side":"central","outcome":"ended in a goal"}),
 ("defender-blocked shots from the left side of the box",     {"side":"left","area":"inside the box","outcome":"was blocked"}),
 ("England Women's goals with the right foot",                {"team":"England Women's","body":"right foot","outcome":"ended in a goal"}),
 ("Denmark shots from outside the area that were blocked",    {"team":"Denmark","area":"outside the box","outcome":"was blocked"}),
 ("wayward headers from the right",                           {"body":"head","side":"right","outcome":"missed the target"}),
 ("Qatar shots saved by the keeper",                          {"team":"Qatar","outcome":"was saved"}),
 ("left-footed attempts inside the 18-yard box that missed",  {"body":"left foot","area":"inside the box","outcome":"missed the target"}),
 ("goals from the right wing with the right foot",            {"side":"right","body":"right foot","outcome":"ended in a goal"}),
 ("Spain Women's headers",                                    {"team":"Spain Women's","body":"head"}),
 ("shots taken centrally from range that were saved",         {"side":"central","area":"outside the box","outcome":"was saved"}),
 ("shots from the left with the right foot that went in",     {"side":"left","body":"right foot","outcome":"ended in a goal"}),
 ("headers blocked on the right of the area",                 {"body":"head","side":"right","area":"inside the box","outcome":"was blocked"}),
 ("Denmark long shots that missed the target",                {"team":"Denmark","area":"outside the box","outcome":"missed the target"}),
 ("England Women's saved shots from the left",                {"team":"England Women's","side":"left","outcome":"was saved"}),
 ("right-footed chances from the centre of the box",          {"body":"right foot","side":"central","area":"inside the box"}),
 ("Qatar goals",                                              {"team":"Qatar","outcome":"ended in a goal"}),
 ("left-foot efforts from the right side that the keeper held", {"body":"left foot","side":"right","outcome":"was saved"}),
 ("headed goals from the left inside the area",               {"body":"head","side":"left","area":"inside the box","outcome":"ended in a goal"}),
 ("Spain Women's blocked shots",                              {"team":"Spain Women's","outcome":"was blocked"}),
 ("shots from far out on the right",                          {"area":"outside the box","side":"right"}),
 ("saved left-footers",                                       {"body":"left foot","outcome":"was saved"}),
 ("Denmark right-foot goals from the centre",                 {"team":"Denmark","body":"right foot","side":"central","outcome":"ended in a goal"}),
]
para_dev, para_test = PARA[0::2], PARA[1::2]      # 18 each: compare models on dev, report on test once

def make_para(split, items):
    docs = [c for c in corpus if c["split"] == split]
    qs   = [{"query": q, "attrs": a} for q, a in items]
    rel  = rel_matrix(docs, [a for _, a in items])
    keep = rel.sum(1) > 0
    for q, k in zip(qs, keep):
        if not k: print("dropped (no relevant doc):", q["query"])
    return docs, [q for q, k in zip(qs, keep) if k], rel[keep]

dev_docs, dev_qs, dev_rel = make_para("val", para_dev)
DEV = (dev_docs, dev_qs, dev_rel)
print(len(dev_qs), "dev paraphrase queries | relevant docs per query:", sorted(dev_rel.sum(1).tolist()))

dropped (no relevant doc): Denmark long shots that missed the target
17 dev paraphrase queries | relevant docs per query: [1, 1, 2, 5, 7, 8, 10, 25, 25, 28, 35, 89, 89, 131, 134, 204, 236]


## 5. Fine-tuning

**Loss: in-batch negatives with false-negative masking.** Each batch of 64 (query, document) pairs gives a 64 x 64 similarity matrix; cross-entropy pushes each query towards its own document. Any other document in the batch that also satisfies the query's attributes is masked (set to -inf), because it is a correct answer, not a negative. The mask comes from `doc_attrs`; it also removes duplicate copies of one shot in a batch.

**Settings:** MiniLM-L6, AdamW lr 2e-5 (1e-5 for the augmented run), weight decay 0.01, batch 64, 10% warm-up then linear decay, gradient clipping 1.0, `max_seq_length` 128.

In [7]:
def attr_codes(d_list, vocab=None):
    """list of attribute dicts -> (n, 5) int tensor; an attribute that is not present -> -1"""
    vocab = vocab or {a: {} for a in ATTRS}
    rows = [[vocab[a].setdefault(d[a], len(vocab[a])) if a in d else -1 for a in ATTRS] for d in d_list]
    return torch.tensor(rows), vocab

def masked_infonce(q, d, qcode, dcode, scale=20.0):
    """q, d: (B, dim) L2-normalised. qcode: attributes asked (B, 5), -1 = not asked. dcode: doc attributes (B, 5)."""
    logits = scale * q @ d.T
    rel = ((qcode[:, None, :] == -1) | (qcode[:, None, :] == dcode[None, :, :])).all(-1)   # doc j satisfies query i
    eye = torch.eye(len(q), dtype=torch.bool, device=q.device)
    logits = logits.masked_fill(rel & ~eye, float("-inf"))      # drop false negatives, keep the true pair
    return F.cross_entropy(logits, torch.arange(len(q), device=q.device))

In [8]:
# ---- training data (template pairs) ----
_, vocab = attr_codes([c["doc_attrs"] for c in corpus])           # one shared value -> int vocabulary
doc_by_shot = {c["shot_id"]: c for c in corpus}
train_pairs = [p for p in pairs if p["split"] == "train"]
tr_q  = [p["query"] for p in train_pairs]
tr_d  = [p["document"] for p in train_pairs]
tr_qc = attr_codes([p["attrs"] for p in train_pairs], vocab)[0].to(device)
tr_dc = attr_codes([doc_by_shot[p["shot_id"]]["doc_attrs"] for p in train_pairs], vocab)[0].to(device)
N = len(train_pairs)
train_shots = [c for c in corpus if c["split"] == "train"]

# ---- query augmentation bank (original template phrases are included in each list) ----
BANK = {
 "body": {"right foot": ["with the right foot", "right-footed", "using the right boot", "struck by the right foot"],
          "left foot":  ["with the left foot", "left-footed", "using the left boot", "hit with the left"],
          "head":       ["with the head", "headed", "as a header", "with a header"],
          "other":      ["with another part of the body", "off a different body part"]},
 "side": {"left":    ["from the left side", "from the left", "down the left flank", "from the left-hand side"],
          "right":   ["from the right side", "from the right", "down the right flank", "from the right-hand side"],
          "central": ["from a central position", "from the centre", "from the center", "from the middle", "straight in front of goal"]},
 "area": {"inside the box":  ["from inside the box", "inside the penalty area", "within the 18-yard box", "in the box", "from close range"],
          "outside the box": ["from outside the box", "from distance", "long-range", "from outside the area", "from range"]},
 "outcome": {"was saved":          ["that was saved", "saved by the goalkeeper", "that the keeper stopped", "that the goalkeeper kept out", "which the keeper saved"],
             "ended in a goal":    ["that ended in a goal", "that scored", "that went in", "that found the net", "that beat the keeper", "resulting in a goal"],
             "was blocked":        ["that was blocked", "blocked by a defender", "that a defender got in the way of", "that was charged down"],
             "missed the target":  ["that missed the target", "that went wide", "that flew over the bar", "that was off target", "that sailed past the post"]},
}
TEAM_FORMS = ["by {}", "from {}", "taken by {}", "for {}"]
OPENERS = ["Find a shot", "Find shots", "Show me shots", "Look for an attempt", "Search for efforts", "Shots", "Attempts"]

def make_aug_queries(doc_attrs, rng, n=3):
    out = []
    for _ in range(n):
        keys = rng.sample(ATTRS, k=rng.choice([2, 3, 3, 4]))
        frags = [rng.choice(TEAM_FORMS).format(doc_attrs["team"]) if k == "team"
                 else rng.choice(BANK[k][doc_attrs[k]]) for k in keys]
        rng.shuffle(frags)
        out.append({"query": rng.choice(OPENERS) + " " + " ".join(frags),
                    "attrs": {k: doc_attrs[k] for k in keys}})
    return out

def build_train(epoch, aug, seed=42):
    """training pairs for one epoch. aug=False -> the original template pairs; aug=True -> fresh augmented queries."""
    if not aug:
        return tr_q, tr_d, tr_qc, tr_dc
    rng = random.Random(seed * 1000 + epoch)                  # new queries every epoch, reproducible
    qs, ds, qa = [], [], []
    for c in train_shots:
        for q in make_aug_queries(c["doc_attrs"], rng):
            qs.append(q["query"]); ds.append(c["document"]); qa.append(q["attrs"])
    qc = attr_codes(qa, vocab)[0].to(device)
    dc = attr_codes([c["doc_attrs"] for c in train_shots for _ in range(3)], vocab)[0].to(device)
    return qs, ds, qc, dc

# ---- helpers ----
def embed(model, texts):
    feats = model.tokenize(texts)
    feats = {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in feats.items()}
    return F.normalize(model(feats)["sentence_embedding"], dim=-1)

@torch.no_grad()
def encode_all(model, texts, bs=256):
    model.eval()
    return torch.cat([embed(model, texts[i:i + bs]) for i in range(0, len(texts), bs)])

def evaluate_model(model, docs, qs, rel):
    D = encode_all(model, [d["document"] for d in docs])
    Q = encode_all(model, [q["query"] for q in qs])
    return evaluate_retrieval((Q @ D.T).cpu().numpy(), rel)

def score_model(m):
    def f(docs, qs):
        D = encode_all(m, [d["document"] for d in docs])
        Q = encode_all(m, [q["query"] for q in qs])
        return (Q @ D.T).cpu().numpy()
    return f

In [9]:
# ---- the training loop. aug=False is "v1" (template queries only); aug=True is "v2" (augmented queries) ----
def fine_tune(lr=2e-5, bs=64, epochs=5, scale=20.0, max_len=128, seed=42, dev=None,
              base="sentence-transformers/all-MiniLM-L6-v2", aug=False, select="para"):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    model = SentenceTransformer(base, device=device)
    model.max_seq_length = max_len
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)

    steps_per_epoch = N // bs
    n_steps, warm = epochs * steps_per_epoch, max(1, int(0.1 * epochs * steps_per_epoch))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / warm if s < warm else max(0.0, (n_steps - s) / (n_steps - warm)))

    g = torch.Generator().manual_seed(seed)
    best, best_state, rows = -1.0, None, []
    for ep in range(1, epochs + 1):
        Qt, Dt, QC, DC = build_train(ep, aug, seed)
        model.train()
        perm, total = torch.randperm(len(Qt), generator=g), 0.0
        for i in range(steps_per_epoch):
            idx = perm[i * bs:(i + 1) * bs]
            q = embed(model, [Qt[j] for j in idx.tolist()])
            d = embed(model, [Dt[j] for j in idx.tolist()])
            loss = masked_infonce(q, d, QC[idx.to(device)], DC[idx.to(device)], scale)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step()
            total += loss.item()
        m = evaluate_model(model, val_docs, val_qs, val_rel).loc["all"]            # template val
        row = {"epoch": ep, "train_loss": total / steps_per_epoch, **{f"tmpl_{k}": v for k, v in m.items()}}
        if dev is not None:                                                         # paraphrase dev
            pm = evaluate_model(model, *dev).loc["all"]
            row.update({f"para_{k}": v for k, v in pm.items()})
        rows.append(row)
        score = row["para_nP@10"] if (dev is not None and select == "para") else row["tmpl_nP@10"]
        print(f"epoch {ep}: loss {row['train_loss']:.3f} | template nP@10 {row['tmpl_nP@10']:.3f}"
              + (f" | paraphrase nP@10 {row['para_nP@10']:.3f} MRR {row['para_MRR']:.3f}" if dev is not None else ""))
        if score > best:                                   # strict >: on ties the earlier epoch is kept
            best = score
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)                      # keep the best epoch, not the last
    return model, pd.DataFrame(rows)

# ---- cache: train once, then load from disk ----
def get_model(name, **kw):
    """Load ../models/<name> if it exists; otherwise train once, save the model and its epoch table.
    To retrain on purpose, delete the folder ../models/<name> and the file ../results/phase3/<name>_hist.csv."""
    path, hist_path = MODELS / name, RES / f"{name}_hist.csv"
    if path.exists() and hist_path.exists():
        print(f"loading saved model: {path}")
        m = SentenceTransformer(str(path), device=device)
        m.max_seq_length = 128
        return m, pd.read_csv(hist_path)
    print(f"no saved model for '{name}': training (this happens once)")
    m, hist = fine_tune(**kw)
    m.save(str(path)); hist.to_csv(hist_path, index=False)
    return m, hist

### v1: template queries only
Trained on the 21,870 template pairs. The best epoch is chosen by **template** val nP@10 (ties keep the earliest epoch); the paraphrase dev scores are logged but not used for selection.

In [10]:
# [TRAINS ONLY THE FIRST TIME (~3 min); afterwards loads from ../models/minilm_v1]
model_v1, hist_v1 = get_model("minilm_v1", dev=DEV, select="tmpl")
hist_v1.round(3)

no saved model for 'minilm_v1': training (this happens once)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3041.75it/s]
The `tokenize` method is deprecated, please use `preprocess` instead.


epoch 1: loss 0.709 | template nP@10 1.000 | paraphrase nP@10 0.653 MRR 0.685
epoch 2: loss 0.016 | template nP@10 1.000 | paraphrase nP@10 0.659 MRR 0.689
epoch 3: loss 0.012 | template nP@10 1.000 | paraphrase nP@10 0.665 MRR 0.686
epoch 4: loss 0.011 | template nP@10 1.000 | paraphrase nP@10 0.671 MRR 0.687
epoch 5: loss 0.010 | template nP@10 1.000 | paraphrase nP@10 0.671 MRR 0.687


Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 13.50it/s]


,epoch,train_loss,tmpl_R@1,tmpl_R@5,tmpl_R@10,tmpl_MRR,tmpl_P@10,tmpl_nP@10,para_R@1,para_R@5,para_R@10,para_MRR,para_P@10,para_nP@10
0,1,0.709,1.0,1.0,1.0,1.0,0.831,1.0,0.647,0.706,0.706,0.685,0.506,0.653
1,2,0.016,1.0,1.0,1.0,1.0,0.831,1.0,0.647,0.706,0.706,0.689,0.512,0.659
2,3,0.012,1.0,1.0,1.0,1.0,0.831,1.0,0.647,0.706,0.706,0.686,0.518,0.665
3,4,0.011,1.0,1.0,1.0,1.0,0.831,1.0,0.647,0.706,0.706,0.687,0.524,0.671
4,5,0.010,1.0,1.0,1.0,1.0,0.831,1.0,0.647,0.706,0.706,0.687,0.524,0.671


**v1 result:** every template-val metric is 1.0 from epoch 1 (train loss 0.71 to 0.01), the same as the oracle. Checks that it is genuine and not a leak: only 5 of 4,389 queries are below 1.0 (worst nP@10 0.8), 0 val documents also appear in train, and retrieved examples really match the asked attributes. The benchmark is simply too easy: documents and queries come from the same fixed templates, so the model only had to learn a lookup. The paraphrase set below tests whether it learned more than that.

In [11]:
systems = {
    "random":       lambda docs, qs: np.random.rand(len(qs), len(docs)),
    "BM25":         bm25_scores,
    "pretrained":   score_model(pre),
    "fine-tuned v1": score_model(model_v1),
}
def compare(docs, qs, rel, systems=systems):
    return pd.DataFrame({name: evaluate_retrieval(fn(docs, qs), rel).loc["all"] for name, fn in systems.items()}).T.round(3)

print(compare(dev_docs, dev_qs, dev_rel))

                 R@1    R@5   R@10    MRR   P@10  nP@10
random         0.000  0.118  0.176  0.062  0.053  0.053
BM25           0.176  0.588  0.588  0.351  0.241  0.315
pretrained     0.176  0.412  0.588  0.310  0.194  0.281
fine-tuned v1  0.647  0.706  0.706  0.685  0.506  0.653


### Error analysis (paraphrase dev)

Count wrong attributes only in the top `min(10, n_relevant)` slots of each query. Using the plain top 10 overstates errors, because a query with 2 relevant documents must fill 8 slots with irrelevant ones.

In [12]:
def error_report(model, docs=dev_docs, qs=dev_qs, rel=dev_rel):
    S = score_model(model)(docs, qs)
    miss, conf, bad = {a: [0, 0] for a in ATTRS}, collections.Counter(), collections.Counter()
    for i, q in enumerate(qs):
        k = min(10, int(rel[i].sum()))
        for j in S[i].argsort()[::-1][:k]:
            for a, v in q["attrs"].items():
                miss[a][1] += 1
                if docs[j]["doc_attrs"][a] != v:
                    miss[a][0] += 1; bad[(a, q["query"])] += 1
                    if a == "outcome": conf[(v, docs[j]["doc_attrs"][a])] += 1
    for a, (w, t) in miss.items():
        print(f"{a:8s} wrong {w}/{t} = {w / max(t, 1):.2f}")
    print()
    for (asked, got), c in conf.most_common(): print(f"asked {asked:18s} got {got:18s} x{c}")
    print()
    for (a, qtext), c in bad.most_common(): print(f"{a:8s} x{c:2d} | {qtext}")

error_report(model_v1)

team     wrong 0/30 = 0.00
body     wrong 1/87 = 0.01
side     wrong 15/74 = 0.20
area     wrong 0/47 = 0.00
outcome  wrong 36/114 = 0.32

asked was saved          got ended in a goal    x18
asked ended in a goal    got missed the target  x17
asked ended in a goal    got was saved          x1

outcome  x10 | right foot shots from the centre that were stopped by the goalkeeper
outcome  x10 | central headers that ended up in the net
side     x10 | right-footed chances from the centre of the box
outcome  x 8 | shots from the left with the right foot that went in
outcome  x 6 | shots the keeper stopped from the middle of the pitch area
side     x 4 | right foot shots from the centre that were stopped by the goalkeeper
outcome  x 2 | left-foot efforts from the right side that the keeper held
body     x 1 | headers from the left wing the keeper saved
side     x 1 | shots the keeper stopped from the middle of the pitch area


### v2: augmented queries
Same loop, but every epoch builds new training queries from the train shots' full attributes with `BANK` (2 to 4 attributes, random order, varied openers; the original template phrases are included). The best epoch is chosen by paraphrase-dev nP@10.

**Contamination caveat:** `BANK` was written after seeing which dev wordings failed, so `para_dev` is now optimistic and `para_test` is partly exposed. The honest number for the report is on fresh queries written after the model is frozen (section 6).

In [13]:
# [TRAINS ONLY THE FIRST TIME (~2 min); afterwards loads from ../models/minilm_aug]
model_aug, hist_aug = get_model("minilm_aug", aug=True, epochs=3, lr=1e-5, dev=DEV)
hist_aug[["epoch", "train_loss", "tmpl_nP@10", "para_nP@10", "para_MRR", "para_R@10"]].round(3)

no saved model for 'minilm_aug': training (this happens once)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3951.50it/s]


epoch 1: loss 1.088 | template nP@10 0.992 | paraphrase nP@10 0.874 MRR 0.956
epoch 2: loss 0.142 | template nP@10 0.999 | paraphrase nP@10 0.993 MRR 1.000
epoch 3: loss 0.059 | template nP@10 0.999 | paraphrase nP@10 1.000 MRR 1.000


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  8.92it/s]


,epoch,train_loss,tmpl_nP@10,para_nP@10,para_MRR,para_R@10
0,1,1.088,0.992,0.874,0.956,1.0
1,2,0.142,0.999,0.993,1.000,1.0
2,3,0.059,0.999,1.000,1.000,1.0


In [14]:
systems["fine-tuned v2 (aug)"] = score_model(model_aug)
print(compare(dev_docs, dev_qs, dev_rel))     # dev is optimistic for v2 (see the caveat above)
error_report(model_aug)

                       R@1    R@5   R@10    MRR   P@10  nP@10
random               0.000  0.118  0.176  0.062  0.053  0.053
BM25                 0.176  0.588  0.588  0.351  0.241  0.315
pretrained           0.176  0.412  0.588  0.310  0.194  0.281
fine-tuned v1        0.647  0.706  0.706  0.685  0.506  0.653
fine-tuned v2 (aug)  1.000  1.000  1.000  1.000  0.788  1.000
team     wrong 0/30 = 0.00
body     wrong 0/87 = 0.00
side     wrong 0/74 = 0.00
area     wrong 0/47 = 0.00
outcome  wrong 0/114 = 0.00




## 6. Final evaluation 


In [18]:
#FRESH_NEAR = [""]    # (query, attrs) tuples, ordinary phrasing not in BANK/PARA. Write AFTER the model is frozen.
#FRESH_FAR  = []    # (query, attrs) tuples, unusual wording / slang / longer sentences.

FRESH_NEAR = [
 ("left-sided efforts the goalkeeper saved",                          {"side":"left","outcome":"was saved"}),
 ("goals from inside the penalty box off the right foot",             {"area":"inside the box","body":"right foot","outcome":"ended in a goal"}),
 ("Denmark shots that hit a defender",                                {"team":"Denmark","outcome":"was blocked"}),
 ("England Women's headed chances out on the right",                  {"team":"England Women's","body":"head","side":"right"}),
 ("shots from 30 yards out that missed the goal altogether",          {"area":"outside the box","outcome":"missed the target"}),
 ("Qatar goals struck with the left foot",                            {"team":"Qatar","body":"left foot","outcome":"ended in a goal"}),
 ("Spain Women's shots straight down the middle",                     {"team":"Spain Women's","side":"central"}),
 ("headers inside the six-yard box that the goalkeeper parried",      {"body":"head","area":"inside the box","outcome":"was saved"}),
 ("right-footed shots from the left that were deflected away by defenders", {"body":"right foot","side":"left","outcome":"was blocked"}),
 ("right-sided goals",                                                {"side":"right","outcome":"ended in a goal"}),
 ("Denmark attempts from outside the 18-yard line that were saved",   {"team":"Denmark","area":"outside the box","outcome":"was saved"}),
 ("central strikes with the head that ended in goals",                {"side":"central","body":"head","outcome":"ended in a goal"}),
 ("England Women's blocked left-footed strikes",                      {"team":"England Women's","outcome":"was blocked","body":"left foot"}),
 ("shots from the right that the goalkeeper got a hand to",           {"side":"right","outcome":"was saved"}),
 ("Qatar attempts inside the area off the head",                      {"team":"Qatar","area":"inside the box","body":"head"}),
 ("right-footed shots from the left that found the back of the net",  {"body":"right foot","side":"left","outcome":"ended in a goal"}),
 ("Spain Women's efforts from beyond the box that missed",            {"team":"Spain Women's","area":"outside the box","outcome":"missed the target"}),
 ("left-footed attempts from a central spot in the box",             {"body":"left foot","side":"central","area":"inside the box"}),
]
FRESH_FAR = [
 ("nodded home from a cross on the left",                             {"body":"head","outcome":"ended in a goal","side":"left"}),
 ("Qatar efforts that the custodian palmed away",                     {"team":"Qatar","outcome":"was saved"}),
 ("wellies from way out that sailed into the stands",                 {"area":"outside the box","outcome":"missed the target"}),
 ("tap-ins from point-blank range",                                   {"area":"inside the box","outcome":"ended in a goal"}),
 ("England Women's blasts that got smothered by a defender's body",  {"team":"England Women's","outcome":"was blocked"}),
 ("bullet headers that crashed off the woodwork",                     {"body":"head","outcome":"missed the target"}),
 ("toe-pokes with the left that were gobbled up by the goalkeeper",   {"body":"left foot","outcome":"was saved"}),
 ("Spain Women's screamers from well outside the box on the right",   {"team":"Spain Women's","area":"outside the box","side":"right"}),
 ("headed attempts that the keeper clawed away from the middle of the box", {"body":"head","side":"central","area":"inside the box","outcome":"was saved"}),
 ("the ones struck with the right peg from out wide on the left that went in", {"body":"right foot","side":"left","outcome":"ended in a goal"}),
 ("Denmark long-rangers that ended up in the net",                    {"team":"Denmark","area":"outside the box","outcome":"ended in a goal"}),
 ("scuffed shots from inside the area that rolled harmlessly past the post", {"area":"inside the box","outcome":"missed the target"}),
 ("headers where a defender threw himself in the way",                {"body":"head","outcome":"was blocked"}),
 ("Qatar volleys with the left foot that the goalkeeper tipped over the bar", {"team":"Qatar","body":"left foot","outcome":"was saved"}),
 ("efforts from the right channel that rattled the net",              {"side":"right","outcome":"ended in a goal"}),
 ("England Women's point-blank headers that were claimed by the keeper", {"team":"England Women's","body":"head","area":"inside the box","outcome":"was saved"}),
 ("swings of the right boot from the centre that cleared the crossbar", {"body":"right foot","side":"central","outcome":"missed the target"}),
 ("Spain Women's headed finishes from the left",                      {"team":"Spain Women's","body":"head","side":"left","outcome":"ended in a goal"}),
]

assert len(FRESH_NEAR) >= 15 and len(FRESH_FAR) >= 15, "write at least 15 queries per tier first"
test_docs, test_qs, test_rel = build_rel("test")

final_systems = {"random": systems["random"], "BM25": bm25_scores, "pretrained": systems["pretrained"],
                 "fine-tuned v1": systems["fine-tuned v1"], "fine-tuned v2 (aug)": systems["fine-tuned v2 (aug)"]}
print("TEMPLATE test queries"); print(compare(test_docs, test_qs, test_rel, final_systems))
for tier, items in (("NEAR", FRESH_NEAR), ("FAR", FRESH_FAR)):
    d, q, r = make_para("test", items)
    print(f"\\nFRESH {tier} paraphrases ({len(q)} queries, test corpus)"); print(compare(d, q, r, final_systems))

TEMPLATE test queries
                       R@1    R@5   R@10    MRR   P@10  nP@10
random               0.071  0.247  0.363  0.163  0.070  0.071
BM25                 0.739  0.959  0.981  0.837  0.643  0.793
pretrained           0.337  0.689  0.797  0.490  0.299  0.397
fine-tuned v1        1.000  1.000  1.000  1.000  0.828  1.000
fine-tuned v2 (aug)  1.000  1.000  1.000  1.000  0.828  1.000
dropped (no relevant doc): England Women's headed chances out on the right
dropped (no relevant doc): Qatar goals struck with the left foot
dropped (no relevant doc): Denmark attempts from outside the 18-yard line that were saved
\nFRESH NEAR paraphrases (15 queries, test corpus)
                       R@1    R@5   R@10    MRR   P@10  nP@10
random               0.067  0.267  0.400  0.169  0.047  0.047
BM25                 0.200  0.667  0.867  0.354  0.200  0.327
pretrained           0.200  0.600  0.733  0.422  0.153  0.242
fine-tuned v1        0.533  0.533  0.600  0.556  0.460  0.600
fine-tuned v2 (

In [19]:
d, q, r = make_para("test", FRESH_NEAR + FRESH_FAR)
error_report(model_aug, d, q, r)

dropped (no relevant doc): England Women's headed chances out on the right
dropped (no relevant doc): Qatar goals struck with the left foot
dropped (no relevant doc): Denmark attempts from outside the 18-yard line that were saved
dropped (no relevant doc): nodded home from a cross on the left
dropped (no relevant doc): Qatar efforts that the custodian palmed away
dropped (no relevant doc): Denmark long-rangers that ended up in the net
dropped (no relevant doc): Qatar volleys with the left foot that the goalkeeper tipped over the bar
dropped (no relevant doc): Spain Women's headed finishes from the left
team     wrong 0/46 = 0.00
body     wrong 0/125 = 0.00
side     wrong 0/126 = 0.00
area     wrong 33/98 = 0.34
outcome  wrong 61/219 = 0.28

asked ended in a goal    got missed the target  x30
asked was saved          got ended in a goal    x14
asked missed the target  got ended in a goal    x10
asked was saved          got missed the target  x7

outcome  x10 | shots from 30 yards out th

In [20]:
def per_query_rr(scores, rel):                       # reciprocal rank of the first relevant doc, per query
    ranked = np.take_along_axis(rel, np.argsort(-scores, axis=1), axis=1)
    return 1.0 / (ranked.argmax(1) + 1)

def paired_bootstrap(a, b, n_boot=10000, seed=0):
    d = a - b                                        # per-query difference, paired by query
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(d), size=(n_boot, len(d)))   # resample queries with replacement
    means = d[idx].mean(1)
    return d.mean(), np.percentile(means, [2.5, 97.5])

for tier, items in (("NEAR", FRESH_NEAR), ("FAR", FRESH_FAR), ("NEAR+FAR", FRESH_NEAR + FRESH_FAR)):
    docs, qs, rel = make_para("test", items)
    s_v1 = score_model(model_v1)(docs, qs)
    s_v2 = score_model(model_aug)(docs, qs)
    for name, f in (("nP@10", per_query_np), ("MRR", per_query_rr)):
        diff, (lo, hi) = paired_bootstrap(f(s_v2, rel), f(s_v1, rel))
        print(f"{tier:9s} {name:6s} v2 - v1 = {diff:+.3f}   95% CI [{lo:+.3f}, {hi:+.3f}]   n={len(qs)}")

dropped (no relevant doc): England Women's headed chances out on the right
dropped (no relevant doc): Qatar goals struck with the left foot
dropped (no relevant doc): Denmark attempts from outside the 18-yard line that were saved
NEAR      nP@10  v2 - v1 = +0.200   95% CI [+0.000, +0.400]   n=15
NEAR      MRR    v2 - v1 = +0.249   95% CI [+0.063, +0.479]   n=15
dropped (no relevant doc): nodded home from a cross on the left
dropped (no relevant doc): Qatar efforts that the custodian palmed away
dropped (no relevant doc): Denmark long-rangers that ended up in the net
dropped (no relevant doc): Qatar volleys with the left foot that the goalkeeper tipped over the bar
dropped (no relevant doc): Spain Women's headed finishes from the left
FAR       nP@10  v2 - v1 = +0.246   95% CI [+0.031, +0.477]   n=13
FAR       MRR    v2 - v1 = +0.249   95% CI [+0.049, +0.477]   n=13
dropped (no relevant doc): England Women's headed chances out on the right
dropped (no relevant doc): Qatar goals struck w